## Example 1: google_api_python_client functionality test 1

In [ ]:
import googleapiclient
import importlib.metadata as im
import pkgutil, importlib

# The distribution is `google-api-python-client`; its import name is `googleapiclient`.
print("google-api-python-client version:", im.version("google-api-python-client"))
print("googleapiclient package:", googleapiclient.__file__)

# Submodule smoke sweep: import every googleapiclient.* submodule (no network).
failures = []
for mod in pkgutil.iter_modules(googleapiclient.__path__, "googleapiclient."):
    try:
        importlib.import_module(mod.name)
    except Exception as e:
        failures.append((mod.name, f"{type(e).__name__}: {e}"))
        raise

print("submodule import failures:", failures or "none")
assert not failures, f"submodule import failures: {failures}"
print("\nExample 1 finished")

## Example 2: google_api_python_client functionality test 2

In [ ]:
from googleapiclient import discovery

# discovery is the primary entrypoint of google-api-python-client.
print("discovery.build:", callable(discovery.build))
print("discovery.build_from_document:", callable(discovery.build_from_document))

for name in ["build", "build_from_document", "Resource"]:
    assert hasattr(discovery, name), f"discovery missing {name}"
    print(f"  discovery.{name}: {getattr(discovery, name)}")

print("\nExample 2 finished")

## Example 3: google_api_python_client functionality test 3

In [ ]:
from googleapiclient import discovery

# Build a service entirely offline from an inline REST discovery document (no network).
doc = {
    "kind": "discovery#restDescription", "name": "smoke", "version": "v1",
    "rootUrl": "https://example.invalid/", "servicePath": "",
    "baseUrl": "https://example.invalid/",
    "schemas": {"Item": {"id": "Item", "type": "object",
                         "properties": {"id": {"type": "string"}, "name": {"type": "string"}}}},
    "resources": {"items": {"methods": {
        "get": {"id": "smoke.items.get", "httpMethod": "GET", "path": "items/{id}",
                "parameters": {"id": {"type": "string", "required": True, "location": "path"}},
                "response": {"$ref": "Item"}}}}},
}
service = discovery.build_from_document(doc)
request = service.items().get(id="abc")
print("built service:", type(service).__name__)
print("request method:", request.method)
print("request uri:", request.uri)
assert request.method == "GET"
assert request.uri.endswith("items/abc?alt=json")
print("\nExample 3 finished")

## Example 4: google_api_python_client functionality test 4

In [ ]:
from googleapiclient.errors import HttpError, UnknownApiNameOrVersion
from googleapiclient.http import HttpRequest, HttpMock

# Exercise the error + http transport surface without any network call.
print("HttpError:", HttpError)
print("HttpRequest:", HttpRequest)
print("HttpMock:", HttpMock)

class _Resp:
    status = 404
    reason = "Not Found"

err = HttpError(_Resp(), b'{"error": {"message": "nope"}}', uri="https://example.invalid/x")
print("HttpError str:", str(err))
assert "404" in str(err)
assert issubclass(UnknownApiNameOrVersion, Exception)
print("\nExample 4 finished")

## Example 5: google_api_python_client functionality test 5

In [ ]:
import urllib.parse  # googleapiclient.model uses urllib.parse.urlencode
from googleapiclient.model import JsonModel

# JsonModel drives request/response (de)serialization — exercise it offline.
model = JsonModel(data_wrapper=False)
headers, params, query, body = model.request(
    headers={}, path_params={"name": "x"}, query_params={}, body_value={"id": "1"}
)
print("serialized request body:", body)
assert body == '{"id": "1"}'
assert headers["content-type"] == "application/json"

class _Resp:
    status = 200

parsed = model.response(_Resp(), '{"id": "1", "name": "widget"}')
print("parsed response:", parsed)
assert parsed["name"] == "widget"
print("\nExample 5 finished")